In [48]:
import json
import math
from functools import reduce

In [49]:
with open('walk_22-guard-else-is-not-what-you-think.json', 'r') as file:
    # data = json.loads(file)
    contents = file.read()
    data = json.loads(contents)
coords = data['features'][0]['geometry']['coordinates']
p1 = { "longitude": coords[0][0], "latitude": coords[0][1] }
p2 = { "longitude": coords[1][0], "latitude": coords[1][1] }
p3 = { "longitude": coords[2][0], "latitude": coords[2][1] }
p4 = { "longitude": coords[3][0], "latitude": coords[3][1] }
p5 = { "longitude": coords[4][0], "latitude": coords[4][1] }

In [50]:
# Some constants are defined for use in the entire module.

# Terrain coeffcients to characterize walking surface.
TERRAIN_COEFFCIENTS = {
    "BLACKTOP": 1.0, # Paved road / treadmill
    "DIRT": 1.1,     # Dirt path, packed trail
    "LIGHT": 1.2,    # Light off-trail, grass
    "SOFT": 1.5,     # Soft sand, deep grass, loose gravel
    "HEAVY": 1.8     # Snow, heavy brush, swamp
}

# Defaults for smoothing out jittery GPS elevation data.
SMOOTH_DEFAULT = True
SMOOTH_DEFAULT_WINDOW = 5

# Maximum plausible walking speed (m/s).
# Speeds higher than this are clamped to this value.
MAX_SPEED_MS = 4.0

# Minimum segment distance. Filters out GPS jitter.
MIN_SEGMENT_DIST_M = 0.5

# Conversion: 1Kcal = 4184 joules
JOULES_PER_KCAL = 4184

# Minimum Mechanics derived constants:
# Table 4, Ludlow & Weyland 2017
MM_COEFFICIENTS = {
    "C1": 0.32,              # grade influence on minimum walking metabolic rate
    "C2": 0.19,              # grade influence on speed-dependent walking metabolic rate
    "C3": 2.66,              # velocity squared coefficient
    "VO2_WALK_MIN": 3.28,    # ml O2 kg-total^-1 min^-1, minimum walking metabolic rate
    "C_DECLINE": 0.73        # fraction of level-grade walking cost applied in decline
}

# Mean measured supine resting metablic rate across all 32 study subjects (ml O2 kg-body^-1 min^-1).
# Used as the default VO2-rest term if no subject-specific resting metabloic rate is given.
# Ludlow & Weyland 2017
DEFAULT_RESTING_VO2 = 3.05

# Standard caloric equivalent of oxygen: ~5kcal per liter O2 per 1000ml.  Expressed here per ml
# for direct multiplication against VO2 rates in ml O2 min^-1.
KCAL_PER_ML_O2 = 0.005

In [51]:
def m2m(milliseconds: int) -> int:
    """Convert a number of milliseconds to minutes.

    Args:
        milliseconds (int): Time in milliseconds.

    Returns:
        int: Time converted into minutes.
    """
    return milliseconds / 60000


In [52]:
def rads(degrees: float) -> float:
    """Convert compass degrees to radians.

    Args:
        degrees (float): Compass degress value.

    Returns:
        float: The calculated radians value.
    """
    return degrees * (math.pi / 180)


In [53]:
def degs(radians: float) -> float:
    """Convert radians to compass degrees.

    Args:
        radians (float): Radians value

    Returns:
        float: The calculated degrees value.
    """
    return radians * (180 / math.pi)


In [54]:
def pointDistance(p1: dict, p2: dict, u = "metric") -> float:
    """Calculate the Haversine distance between two GPS points.

    Args:
        p1 (dict): Dictionary containing latitude and longitude values.
        p2 (dict): Dictionary containing latitude and longitude values.
        u (string): String value indicating unit system to use.

    Returns:
        float: The Haversine distance between GPS points p1 and p2.

    Raises:
        ValueError: if p1 argument is missing longitude or latitude values.
        ValueError: if p2 argument is missing longitude or latitude values.
    """
    if "longitude" not in p1 or "latitude" not in p1:
        raise ValueError(f"Point p1 argument is requires longitude and latitude values.")
    if "longitude" not in p2 or "latitude" not in p2:
        raise ValueError(f"Point p2 argument is requires longitude and latitude values.")
    earthRadiusKm = 6371
    earthRadiusMeters = 6371000
    earthRadiusMiles = 3959
    _u = u.lower()
    r = None
    if _u == 'm' or _u == 'meters':
        r = earthRadiusMeters
    elif _u == 'km' or _u == 'kilometers':
        r = earthRadiusKm
    elif _u == 'mi' or _u == 'miles' or _u == 'imperial':
        r = earthRadiusMiles
    else:
        r = earthRadiusMeters
    #print(r, _u)
    dLat = rads(p2['latitude'] - p1['latitude'])
    dLon = rads(p2['longitude'] - p1['longitude'])
    lat1 = rads(p1['latitude'])
    lat2 = rads(p2['latitude'])
    a = math.sin(dLat / 2) * math.sin(dLat / 2) \
        + math.sin(dLon / 2) * math.sin(dLon / 2) * math.cos(lat1) * math.cos(lat2)
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return c * r


print(pointDistance(p1, p2, u ='mi'))
print(pointDistance(p1, p3, u ='mi'))
print(pointDistance(p1, p4, u ='mi'))
print(pointDistance(p1, p5, u ='mi'))
# p6 = { "latitude": 10 }
# print(pointDistance(p1, p6, u ='mi'))
# help(pointDistance)

0.00010394011574007407
0.00021639922403606193
0.0003388536049015972
0.0026016707988299878


In [55]:
# Calculate the difference in altitude between two points.
def calculateVerticalInterval(alt1: float, alt2: float) -> float:
    """Calculate the difference in altitude between two points.

    Args:
        alt1 (float): First altitude value.
        alt2 (float): Second altitude value.

    Returns:
        float: Altitude difference.
    """
    return alt2 - alt1


In [56]:
# Calculate the slope between two points.
def calculateSlopeGrade(point1: dict, point2: dict) -> dict:
    """Calcuate the slope between two GPS points.

    Args:
        point1 (dict): Dictionary with longitude and latitude properties.
        point2 (dict): Dictionary with longitude and latitude properties.

    Returns:
        dict: Dictionary with grade and angleDegrees properties.
    """
    horizontalDistance = pointDistance(point1, point2)
    verticalDistance = calculateVerticalInterval(point1["altitude"], point2["altitude"])
    if horizontalDistance == 0:
        return { "grade": math.inf, "angleDegrees": 90 }
    slope = verticalDistance / horizontalDistance
    grade = slope * 100
    angle = math.atan(slope) * 180
    return {
        "grade": grade,
        "angleDegrees": angle / math.pi
    }

In [57]:
# Apply a simple rolling-average smoother to the altitude values in a coordinates array.
# Raw GPS altitude can have +-5 to 15 m of noise, which can create artificial grade spikes
# that inflate calorie estimates.
def smoothAltitude(coords: List[List[float]], windowSize: int = SMOOTH_DEFAULT_WINDOW) -> List[List[float]]:
    """Apply a simple rolling-average smoothing function to the altitude values in a coordinates array.

    Args:
        coords (List[List[float]]: List of coordinate arrays.
        windowSize (int): Number of points to average (odd number recommended).

    Returns:
        List[List[float]]: New coordinates array with smoothed altitudes.
    """
    half = math.floor(windowSize / 2)
    print(f"windowSize: {windowSize}, half: {half}")
    smoothed = list()
    i = 0
    n = len(coords)
    print(f"coords length: {n}")
    while i < n:
        print(f"\tstarting loop: {i}")
        start = max(0, i - half)
        end = min(n - 1, (i + half) + 2) # + 2 because slice end index is non-inclusive
        print(f"\tstart: {start}, end: {end}")
        slice = coords[start:end]
        print(f"\tslice (length {len(slice)}): {slice}\n")
        validAlts = [x[3] for x in slice if x[3] is not None]
        # print(f"\tvalidAlts: {validAlts}")
        averageAltitude = reduce(lambda acc, curr: acc + curr, validAlts, 0) / len(validAlts) if len(validAlts) > 0 else slice[3]
        smoothed.append([coords[i][0], coords[i][1], coords[i][2], averageAltitude, coords[i][4], coords[i][5]])
        print(f"\tsmoothed altitude: {averageAltitude}, {validAlts}\n")
        i = i + 1
    return smoothed

# coordinateSlice = coords[0:25]
# print(coordinateSlice)
# print(smoothAltitude(coordinateSlice))

In [58]:
# Simple MET based calorie estimate.
def simpleCalories(minutes: int = 1, weights: dict = { "body": 0, "ruck": 0, "water": 0 }, MET: float = 7.5) -> float:
    """The simplest calorie estimating function.  Calculates the ratio of energy spent per unit time during a specific
    physical activity to a reference value of 3.5 ml O2 / (kg·min).

    Args:
        minutes (float): Time spent expending energy, in minutes.
        weights (dict): Collection of weight values, in kilograms.
        MET (float): The Metabolic Equivalent Task number of activity.

    Raises:
        ValueError: If minutes is not a valid, positive number.
        ValueError: If weights.body is not a valid, positive number.
        ValueError: If MET is not a valid, positive number.
        
    Returns:
        float: Number of calories burned.
    """
    if minutes <= 0 or minutes is None:
        raise ValueError(f"Minutes must be a positive, finite number. (Supplied {minutes})")
    if weights["body"] <= 0 or weights["body"] is None:
        raise ValueError(f"Body weight must be a positive, finite number.  (Supplied {weights["body"]}")
    if MET <= 0 or MET is None:
        raise ValueError(f"MET must be a positive, finite number.  (Supplied {MET})")
    COMBINED = weights["body"] + weights["ruck"] + weights["water"]
    # print(COMBINED)
    return ((MET * 3.5 * COMBINED) / 200) * minutes
    
# simpleCalories(0, {"body": 10}, MET = 7.5)
# simpleCalories(10, {"body": 0}, MET = 7.5)
# simpleCalories(10, {"body": 60}, MET = -1)
print(simpleCalories(10, {"body": 60, "ruck": 5, "water": 1}, 7.5))

86.625


In [59]:
# Corrective factor for downhill (G < 0) segments of the hike.
def santeeCorrective(W: float, L: float, V: float, G: float, n: float) -> float:
    """Corrective factor for downhill (G < 0) segments of the hike.

    Args:
        W (float): Body weight measured in kg.
        L (float): Load weight measured in kg.
        V (float): Walking speed in m/s.
        G (float): Hill grade as a percentage (e.g 10 for 10% incline, -5 for decline).
        n (float): Terrain characterization coefficient.

    Returns:
        float: Downhill corrective factor in Watts.
    """
    return n * ( \
        (G * (W + L) * V) / 3.5 \
        - ((W + L) * (((G + 6) ** 2) / W)) \
        + (25 * (V ** 2)) \
    )


In [60]:
# Calculate the metabolic rate (Watts) using Pandolf-Santee predictive model.
def pandolfMetabolicRate(W: float, L: float, V: float, G: float, n: float) -> float:
    """Calculate the metabolic rate (Watts) using Pandolf-Santee predictive model.

    Args:
        W (float): Body weight measured in kg.
        L (float): Load weight measured in kg.
        V (float): Walking speed in m/s.
        G (float): Hill grade as a percentage (e.g 10 for 10% incline, -5 for decline).
        n (float): Terrain characterization coefficient.

    Returns:
        float: Metabolic rate in Watts (should always be >= 0).
    """
    if V <= 0:
        return 0
    loadRatio = L / W
    M = 1.5 * W \
        + 2 * (W + L) * loadRatio ** 2 \
        + n * (W + L) * (1.5 * V ** 2 + 0.35 * V  * G)
    correction = 0
    if G < 0:
        correction = santeeCorrective(W, L, V, G, n)
    # the equation can return negative values on steep descents so clamp to 0.
    return max(0, M - correction)


In [77]:
# Use the Pandolf-Santee predictive model to calculate the total (and per-segment) calorie expenditure for a GPS track.
def pandolfCalories(coords: List[List[float]] = [], options: dict = {}) -> dict:
    """Use the Pandolf-Santee predictive model to calculate the total (and per-segment) calorie expenditure for a GPS track.

    Args:
        coords (List[List[float]]): GPS coordinates array.  Each element:
            [longitude, latitude, heading, altitude (m), accuracy (m), timestamp (ms)]
        options (dict): Options
        options["bodyWeightKg"] (float): Body weight in kg (required).
        options["loadKg"] = 0 (float): Load/pack weight in kg (optional).
        options["waterKg"] = 0 (float): Water weight in kg carried (optional).
        options["terrain"] = 1.1 (float): Terrain coefficient (optional).  Use TERRAIN_COEFFICIENTS.
        options["smooth"] = True (Boolean): Whether to smooth GPS altitude values (optional).
        options["smoothWindow"] = 5 (int): Rolling average window size for smoothing (optional).
        options["returnSegments"] = False (Boolean): Return array of all segments calculated (optional)?

    Raises:
        ValueError: If coords array contains less than 2 items.
        ValueError: If required body weight is < 0, null, or otherwise invalid.

    Returns:
        dict: Results
        {
            totalKcal,        # Total calories burned.
            totalDistanceM,   # Total horizontal distance (meters).
            totalDurationSec, # Total elapsed time (seconds).
            avgSpeedMs,       # Average speed (m/s).
        }
    """
    bodyWeightKg = options.get("bodyWeightKg", 0)
    loadKg = options.get("loadKg", 0)
    waterKg = options.get("waterKg", 0)
    terrain = options.get("terrain", 1.1)
    smooth = options.get("smooth", True)
    smoothWindow = options.get("smoothWindow", SMOOTH_DEFAULT_WINDOW)
    returnSegments = options.get("returnSegments", False)
    if len(coords) < 2:
        raise ValueError(f"The coordinates array needs at least 2 elements, {len(coords)} provided.")
    if not bodyWeightKg or bodyWeightKg <= 0:
        raise ValueError(f"options.bodyWeightkg is required and must be a positive number, {bodyWeightKg} provided.")
    track = smoothAltitude(coords, smoothWindow) if smooth  else coords
    print(len(track))
          
opts = {"bodyWeightKg": 60, "smooth": False}
coo = coords[:50]
pandolfCalories(coo, opts)

50


In [68]:
options = {
    "bodyWeight": 60,
    "ruckWeightKg": 5,
    "waterWeightKg": 0
}
bodyWeightKg = options.get("bodyWeightKg", 10)
print(bodyWeightKg)

10


In [62]:
# dir(math)

In [61]:
smooth = list()
smooth.append('one')
smooth.append('two')
for x in range(10):
    if x % 2 == 0:
        smooth.append(x)
    else:
        smooth.append(None)

print(smooth)
# print(len(smooth))
i = 0
n = len(smooth)
while i < n:
    print(i, smooth[i])
    i += 1

print(smooth[2:6+1])
newSmooth = [x for x in smooth[2:11] if x is not None and x > 1 and x < 8]
print(newSmooth)

['one', 'two', 0, None, 2, None, 4, None, 6, None, 8, None]
0 one
1 two
2 0
3 None
4 2
5 None
6 4
7 None
8 6
9 None
10 8
11 None
[0, None, 2, None, 4]
[2, 4, 6]
